# 08 · Variables operacionales observables a T−60

Objetivo único: predecir `Arrival_Delay_Min` exactamente una hora antes del off-block
programado. Para cada vuelo se define `cutoff = FILED OFF BLOCK TIME - 60 minutos`.
Los retrasos de salida solo se incorporan cuando el off-block real ya ocurrió; los de
llegada solo cuando la llegada real ya ocurrió. El test externo no se lee.

La selección se realiza en una partición temporal interna de train. Validation se usa una
sola vez al final y test permanece reservado.


In [1]:
from functools import reduce
from pathlib import Path
import gc
import importlib.util
import json
import math
import sys
import time

import joblib
import numpy as np
import pandas as pd
import psutil
from scipy import sparse
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from pyspark import StorageLevel
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.flight_config import DataQualityConfig
from src.spark_flight_pipeline import (
    apply_aviation_rules, calculate_delays, create_spark, fill_text_nulls,
    read_flights_spark,
)
from src.t60_modeling import (
    HashedRidgePreprocessor, HistoricalMedianBaseline, add_schedule_features,
    compact_score, ensemble_grid, segment_metrics,
)
from src.t60_operational_features import (
    add_prediction_cutoff, build_rolling_event_features,
    build_rotation_features, deterministic_percent_sample,
)

RUN_EXPERIMENT = True
REBUILD_FEATURE_BLOCKS = False  # Reuse audited Parquet blocks when present.
TARGET = 'Arrival_Delay_Min'
TRAIN_SAMPLE_PERCENT = 10
VALIDATION_SAMPLE_PERCENT = 5
WINDOWS_HOURS = (1, 6, 24)
INTERNAL_TUNING_START = pd.Timestamp('2022-09-01 00:00:00')
MIN_AVAILABLE_RAM_GB = 5.0
EXPECTED_TRAIN_ROWS = 2_457_169
EXPECTED_VALIDATION_ROWS = 591_391
EXPECTED_TRAIN_SAMPLE_ROWS = 245_590
EXPECTED_VALIDATION_SAMPLE_ROWS = 29_315
GLOBAL_GUARDRAIL_MINUTES = 0.25
REQUIRED_COMBINED_IMPROVEMENT = 0.20

MODEL_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'model' / 'arrival_pre'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'model' / 'arrival_pre_t60_ops_10pct'
BLOCK_ROOT = OUTPUT_ROOT / 'feature_blocks'
REPORT_ROOT = PROJECT_ROOT / 'reports'
MODELS_ROOT = PROJECT_ROOT / 'models'
ABLATION_REPORT = REPORT_ROOT / '08_t60_feature_ablation.csv'
TUNING_REPORT = REPORT_ROOT / '08_t60_internal_tuning.csv'
ENSEMBLE_REPORT = REPORT_ROOT / '08_t60_ensemble_grid.csv'
COMPARISON_REPORT = REPORT_ROOT / '08_t60_operational_model_comparison.csv'
PREDICTIONS_PATH = REPORT_ROOT / '08_t60_validation_predictions.parquet'
DECISION_PATH = REPORT_ROOT / '08_t60_scaling_decision.json'


## Preflight y contrato de ejecución

Solo se leen los Parquet `train` y `validation`. Del histórico crudo se excluye explícitamente
marzo de 2023, que corresponde al periodo de test. Los bloques operacionales se materializan
en Parquet para evitar repetir las ventanas Spark.


In [2]:
memory = psutil.virtual_memory()
available_ram_gb = memory.available / (1024 ** 3)
assert abs((INTERNAL_TUNING_START - pd.Timestamp('2022-09-01')).total_seconds()) < 1
assert importlib.util.find_spec('catboost') is not None
assert MODEL_ROOT.joinpath('train').exists()
assert MODEL_ROOT.joinpath('validation').exists()
if RUN_EXPERIMENT and available_ram_gb < MIN_AVAILABLE_RAM_GB:
    raise RuntimeError(
        f'Solo hay {available_ram_gb:.2f} GB libres; se requieren al menos '
        f'{MIN_AVAILABLE_RAM_GB:.1f} GB.'
    )
for path in (OUTPUT_ROOT, BLOCK_ROOT, REPORT_ROOT, MODELS_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print({
    'run': RUN_EXPERIMENT, 'available_ram_gb': round(available_ram_gb, 2),
    'train_percent': TRAIN_SAMPLE_PERCENT,
    'validation_percent': VALIDATION_SAMPLE_PERCENT,
    'windows_hours': WINDOWS_HOURS, 'rebuild_feature_blocks': REBUILD_FEATURE_BLOCKS,
    'test_processed_read': False,
})


{'run': True, 'available_ram_gb': 5.8, 'train_percent': 10, 'validation_percent': 5, 'windows_hours': (1, 6, 24), 'rebuild_feature_blocks': False, 'test_processed_read': False}


## Carga de objetivos e histórico observable

Los objetivos proceden del pipeline congelado. El histórico operativo se reconstruye con las
mismas reglas físicas y de alcance: únicamente vuelos regulares comerciales (`ICAO Flight Type=S`).
`AC Registration` se enlaza por `ECTRL ID`; sus tiempos reales nunca se añaden al vuelo objetivo.


In [3]:
if RUN_EXPERIMENT:
    spark = create_spark(
        'arrival-pre-t60-operational', master='local[2]',
        driver_memory='4g', shuffle_partitions=16,
    )
    spark.sparkContext.setLogLevel('WARN')
    train_base = spark.read.parquet(str(MODEL_ROOT / 'train'))
    validation_base = spark.read.parquet(str(MODEL_ROOT / 'validation'))
    base_counts = {'train': train_base.count(), 'validation': validation_base.count()}
    assert base_counts == {
        'train': EXPECTED_TRAIN_ROWS, 'validation': EXPECTED_VALIDATION_ROWS,
    }
    train_sample = deterministic_percent_sample(
        train_base, TRAIN_SAMPLE_PERCENT
    ).withColumn('_split', F.lit('train'))
    validation_sample = deterministic_percent_sample(
        validation_base, VALIDATION_SAMPLE_PERCENT
    ).withColumn('_split', F.lit('validation'))
    sample_counts = {
        'train': train_sample.count(), 'validation': validation_sample.count(),
    }
    assert sample_counts == {
        'train': EXPECTED_TRAIN_SAMPLE_ROWS,
        'validation': EXPECTED_VALIDATION_SAMPLE_ROWS,
    }, sample_counts

    history_files = [
        str(path) for path in sorted((PROJECT_ROOT / 'data' / 'raw' / 'flights').glob('Flights_*.csv.gz'))
        if '202303' not in path.name
    ]
    assert len(history_files) == 5
    assert not any('202303' in path for path in history_files)
    events = apply_aviation_rules(
        calculate_delays(read_flights_spark(spark, history_files)),
        DataQualityConfig(min_delay_minutes=-120, regular_commercial_only=True),
    )
    events = fill_text_nulls(events, ['AC Operator', 'AC Registration']).filter(
        F.col('FILED OFF BLOCK TIME') < F.to_timestamp(F.lit('2023-01-01 00:00:00'))
    ).select(
        'ECTRL ID', 'ADEP', 'ADES', 'AC Operator', 'AC Registration',
        'FILED OFF BLOCK TIME', 'ACTUAL OFF BLOCK TIME', 'ACTUAL ARRIVAL TIME',
        'Departure_Delay_Min', 'Arrival_Delay_Min',
    ).persist(StorageLevel.DISK_ONLY)
    event_rows = events.count()
    event_ids = events.select('ECTRL ID').distinct().count()
    assert event_rows == event_ids, (event_rows, event_ids)
    registration_stats = events.agg(
        F.countDistinct(F.when(F.col('AC Registration') != 'Unknown', F.col('AC Registration'))).alias('distinct_known'),
        F.avg((F.col('AC Registration') != 'Unknown').cast('double')).alias('known_rate'),
    ).first().asDict()
    registry = events.select('ECTRL ID', 'AC Registration').dropDuplicates(['ECTRL ID'])
    target_union = add_prediction_cutoff(
        train_sample.unionByName(validation_sample).join(registry, 'ECTRL ID', 'left')
        .fillna({'AC Registration': 'Unknown'})
    ).persist(StorageLevel.DISK_ONLY)
    target_rows = target_union.count()
    assert target_rows == sum(sample_counts.values())
    target_registration_rate = target_union.agg(
        F.avg((F.col('AC Registration') != 'Unknown').cast('double')).alias('rate')
    ).first()['rate']
    print({
        'base_counts': base_counts, 'sample_counts': sample_counts,
        'history_files': [Path(path).name for path in history_files],
        'event_rows': event_rows, 'registration_stats': registration_stats,
        'target_registration_rate': round(target_registration_rate, 5),
        'test_processed_read': False,
    })


{'base_counts': {'train': 2457169, 'validation': 591391}, 'sample_counts': {'train': 245590, 'validation': 29315}, 'history_files': ['Flights_20211201_20211231.csv.gz', 'Flights_20220301_20220331.csv.gz', 'Flights_20220601_20220630.csv.gz', 'Flights_20220901_20220930.csv.gz', 'Flights_20221201_20221231.csv.gz'], 'event_rows': 3048560, 'registration_stats': {'distinct_known': 11517, 'known_rate': 0.9998464848977878}, 'target_registration_rate': 0.99987, 'test_processed_read': False}


## Ventanas 1/6/24 horas y rotación

Se calculan conteo, media, desviación y proporción con retraso >15. Estas estadísticas permiten
restar exactamente el vuelo objetivo si una salida extremadamente temprana cayese antes del cutoff.
Cada bloque conserva temporalmente el máximo timestamp utilizado para auditar `evento <= cutoff`.

Bloques: salidas en ADEP, llegadas en ADES, llegadas por ruta y actividad de salida/llegada
del operador. La rotación utiliza la llegada completada más reciente de la misma matrícula.


In [4]:
if RUN_EXPERIMENT:
    feature_specs = [
        ('adep_dep', ['ADEP'], 'ACTUAL OFF BLOCK TIME', 'Departure_Delay_Min'),
        ('ades_arr', ['ADES'], 'ACTUAL ARRIVAL TIME', 'Arrival_Delay_Min'),
        ('route_arr', ['ADEP', 'ADES'], 'ACTUAL ARRIVAL TIME', 'Arrival_Delay_Min'),
        ('operator_dep', ['AC Operator'], 'ACTUAL OFF BLOCK TIME', 'Departure_Delay_Min'),
        ('operator_arr', ['AC Operator'], 'ACTUAL ARRIVAL TIME', 'Arrival_Delay_Min'),
    ]
    audit_path = REPORT_ROOT / '08_t60_feature_leakage_audit.csv'
    if not REBUILD_FEATURE_BLOCKS and audit_path.exists():
        audit_rows = [
            row for row in pd.read_csv(audit_path).to_dict('records')
            if row['block'] != 'rotation'
        ]
    else:
        audit_rows = []
    feature_block_names = []
    for prefix, keys, event_time, value in feature_specs:
        block_path = BLOCK_ROOT / prefix
        if (
            not REBUILD_FEATURE_BLOCKS and audit_path.exists()
            and (block_path / '_SUCCESS').exists()
        ):
            feature_block_names.append(prefix)
            print({'block': prefix, 'reused': True})
            continue
        started = time.perf_counter()
        block = build_rolling_event_features(
            target_union, events, key_columns=keys,
            event_time_column=event_time, value_column=value,
            prefix=prefix, windows_hours=WINDOWS_HOURS,
        ).persist(StorageLevel.DISK_ONLY)
        block_rows = block.count()
        max_time_columns = [f'_{prefix}_{hours}h_max_event_time' for hours in WINDOWS_HOURS]
        violation_condition = reduce(
            lambda left, right: left | right,
            [F.col(column) > F.col('prediction_cutoff_t60') for column in max_time_columns],
        )
        leakage_violations = block.filter(violation_condition).count()
        self_columns = [f'_{prefix}_{hours}h_self_removed' for hours in WINDOWS_HOURS]
        self_removed = block.agg(
            *[F.sum(column).alias(column) for column in self_columns]
        ).first().asDict()
        assert block_rows == target_rows
        assert leakage_violations == 0
        model_columns = [
            column for column in block.columns
            if column == 'ECTRL ID'
            or (not column.startswith('_') and column != 'prediction_cutoff_t60')
        ]
        block.select(*model_columns).write.mode('overwrite').parquet(
            str(BLOCK_ROOT / prefix)
        )
        audit_rows.append({
            'block': prefix, 'rows': block_rows,
            'leakage_violations': leakage_violations,
            'self_contributions_removed': int(sum(value or 0 for value in self_removed.values())),
            'seconds': time.perf_counter() - started,
        })
        feature_block_names.append(prefix)
        block.unpersist()
        print(audit_rows[-1])

    started = time.perf_counter()
    rotation = build_rotation_features(target_union, events).persist(StorageLevel.DISK_ONLY)
    rotation_rows = rotation.count()
    rotation_leakage = rotation.filter(
        F.col('_rotation_previous_event_time') > F.col('prediction_cutoff_t60')
    ).count()
    rotation_self_matches = rotation.agg(F.sum('_rotation_self_match').alias('n')).first()['n'] or 0
    rotation_self_value_leakage = rotation.filter(
        (F.col('_rotation_self_match') == 1)
        & (
            F.col('rotation_previous_arrival_delay').isNotNull()
            | F.col('rotation_previous_departure_delay').isNotNull()
            | F.col('rotation_minutes_since_previous_arrival').isNotNull()
            | (F.col('rotation_history_available') != 0)
        )
    ).count()
    assert rotation_leakage == 0
    assert rotation_self_value_leakage == 0
    rotation.select(
        'ECTRL ID', 'rotation_previous_arrival_delay',
        'rotation_previous_departure_delay',
        'rotation_minutes_since_previous_arrival',
        'rotation_history_available',
    ).write.mode('overwrite').parquet(str(BLOCK_ROOT / 'rotation'))
    audit_rows.append({
        'block': 'rotation', 'rows': rotation_rows,
        'leakage_violations': rotation_leakage,
        'self_contributions_removed': int(rotation_self_matches),
        'seconds': time.perf_counter() - started,
    })
    rotation.unpersist()
    feature_block_names.append('rotation')
    feature_audit_pd = pd.DataFrame(audit_rows)
    feature_audit_pd.to_csv(audit_path, index=False)
    display(feature_audit_pd)


{'block': 'adep_dep', 'reused': True}
{'block': 'ades_arr', 'reused': True}
{'block': 'route_arr', 'reused': True}
{'block': 'operator_dep', 'reused': True}
{'block': 'operator_arr', 'reused': True}


,block,rows,leakage_violations,self_contributions_removed,seconds
0,adep_dep,274905,0,216,176.391220
1,ades_arr,274905,0,0,172.469762
2,route_arr,274905,0,0,51.360237
3,operator_dep,274905,0,216,487.616835
4,operator_arr,274905,0,0,897.581358
5,rotation,274869,0,0,18.248931


## Dataset enriquecido reutilizable

Los bloques auditados se unen uno a uno por `ECTRL ID`. El resultado conserva únicamente
variables programadas, categorías conocidas a T−60 y agregados históricos. No contiene tiempos
reales ni retrasos del vuelo objetivo distintos del target.


In [5]:
if RUN_EXPERIMENT:
    enriched = target_union
    for block_name in feature_block_names:
        block = spark.read.parquet(str(BLOCK_ROOT / block_name))
        block_rows = block.count()
        assert block_rows == target_rows if block_name != 'rotation' else block_rows <= target_rows
        if 'prediction_cutoff_t60' in block.columns:
            block = block.drop('prediction_cutoff_t60')
        enriched = enriched.join(block, 'ECTRL ID', 'left')
    forbidden = {
        'ACTUAL OFF BLOCK TIME', 'ACTUAL ARRIVAL TIME',
        'Departure_Delay_Min', 'Actual Distance Flown (nm)',
    }
    assert not (forbidden & set(enriched.columns))
    train_enriched = enriched.filter(F.col('_split') == 'train').drop('_split')
    validation_enriched = enriched.filter(F.col('_split') == 'validation').drop('_split')
    assert train_enriched.count() == EXPECTED_TRAIN_SAMPLE_ROWS
    assert validation_enriched.count() == EXPECTED_VALIDATION_SAMPLE_ROWS
    train_enriched.write.mode('overwrite').parquet(str(OUTPUT_ROOT / 'train'))
    validation_enriched.write.mode('overwrite').parquet(str(OUTPUT_ROOT / 'validation'))
    print({
        'train_rows': EXPECTED_TRAIN_SAMPLE_ROWS,
        'validation_rows': EXPECTED_VALIDATION_SAMPLE_ROWS,
        'feature_columns': len(train_enriched.columns),
        'output': str(OUTPUT_ROOT), 'test_processed_read': False,
    })
    target_union.unpersist()
    events.unpersist()
    spark.stop()


{'train_rows': 245590, 'validation_rows': 29315, 'feature_columns': 87, 'output': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\data\\processed\\model\\arrival_pre_t60_ops_10pct', 'test_processed_read': False}


## Train interno, tuning temporal y conjuntos de variables

El 10% de train se divide por fecha: meses anteriores a septiembre para ajuste interno y
septiembre de 2022 para tuning. La ablación y los pesos del ensemble se deciden solo aquí.


In [6]:
if RUN_EXPERIMENT:
    train_pd = add_schedule_features(pd.read_parquet(OUTPUT_ROOT / 'train'))
    validation_pd = add_schedule_features(pd.read_parquet(OUTPUT_ROOT / 'validation'))
    train_pd = train_pd.sort_values('FILED OFF BLOCK TIME').reset_index(drop=True)
    validation_pd = validation_pd.sort_values('FILED OFF BLOCK TIME').reset_index(drop=True)
    forbidden = {
        'ACTUAL OFF BLOCK TIME', 'ACTUAL ARRIVAL TIME',
        'Departure_Delay_Min', 'Actual Distance Flown (nm)',
    }
    assert not (forbidden & set(train_pd.columns))
    cutoff_delta = (
        pd.to_datetime(train_pd['FILED OFF BLOCK TIME'])
        - pd.to_datetime(train_pd['prediction_cutoff_t60'])
    ).dt.total_seconds() / 60.0
    assert np.allclose(cutoff_delta, 60.0)
    internal_fit_pd = train_pd[
        pd.to_datetime(train_pd['FILED OFF BLOCK TIME']) < INTERNAL_TUNING_START
    ].copy()
    internal_tuning_pd = train_pd[
        pd.to_datetime(train_pd['FILED OFF BLOCK TIME']) >= INTERNAL_TUNING_START
    ].copy()
    assert len(internal_fit_pd) + len(internal_tuning_pd) == len(train_pd)

    CATEGORICAL_COLUMNS = [
        'ADEP', 'ADES', 'AC Operator', 'AC Type_grouped',
        'STATFOR Market Segment', 'Class_aircraft',
        'Number+Engine Type_aircraft', 'AC Registration',
    ]
    STATIC_NUMERIC_COLUMNS = [
        'Requested_FL_Imputed', 'scheduled_duration_min',
        'departure_hour_sin', 'departure_hour_cos',
        'departure_dow_sin', 'departure_dow_cos', 'departure_month',
    ]
    operational_columns = [
        column for column in train_pd.columns
        if column.startswith(('adep_dep_', 'ades_arr_', 'route_arr_',
                              'operator_dep_', 'operator_arr_', 'rotation_'))
    ]
    feature_groups = {
        'airport': [column for column in operational_columns if column.startswith(('adep_dep_', 'ades_arr_'))],
        'route': [column for column in operational_columns if column.startswith('route_arr_')],
        'operator': [column for column in operational_columns if column.startswith(('operator_dep_', 'operator_arr_'))],
        'rotation': [column for column in operational_columns if column.startswith('rotation_')],
    }
    print({
        'internal_fit_rows': len(internal_fit_pd),
        'internal_tuning_rows': len(internal_tuning_pd),
        'validation_rows': len(validation_pd),
        'operational_features': len(operational_columns),
        'feature_groups': {key: len(value) for key, value in feature_groups.items()},
        'registration_known_rate_train': round((train_pd['AC Registration'] != 'Unknown').mean(), 5),
    })


{'internal_fit_rows': 172315, 'internal_tuning_rows': 73275, 'validation_rows': 29315, 'operational_features': 64, 'feature_groups': {'airport': 24, 'route': 12, 'operator': 24, 'rotation': 4}, 'registration_known_rate_train': np.float64(0.99987)}


## Ablación de entidades y ventanas

Una Ridge con hashing constante compara grupos de variables y la eliminación individual de
1h, 6h y 24h. Se selecciona el menor MAE combinado sujeto al guardrail global de 0,25 minutos.


In [7]:
if RUN_EXPERIMENT:
    hasher = FeatureHasher(n_features=1 << 15, input_type='string', alternate_sign=True)
    def categorical_tokens(frame):
        categorical = frame[CATEGORICAL_COLUMNS].fillna('__MISSING__').astype(str)
        return (
            [f'{column}={value}' for column, value in zip(CATEGORICAL_COLUMNS, row)]
            for row in categorical.itertuples(index=False, name=None)
        )
    cat_fit_matrix = hasher.transform(categorical_tokens(internal_fit_pd))
    cat_tuning_matrix = hasher.transform(categorical_tokens(internal_tuning_pd))

    entity_sets = {
        'static_only': [],
        'airport': feature_groups['airport'],
        'airport_route': feature_groups['airport'] + feature_groups['route'],
        'airport_route_operator': feature_groups['airport'] + feature_groups['route'] + feature_groups['operator'],
        'all_plus_rotation': operational_columns,
    }
    all_entity_columns = entity_sets['all_plus_rotation']
    candidate_sets = dict(entity_sets)
    for hours in WINDOWS_HOURS:
        candidate_sets[f'all_without_{hours}h'] = [
            column for column in all_entity_columns if f'_{hours}h_' not in column
        ]

    ablation_rows = []
    candidate_numeric_columns = {}
    for candidate, extra_columns in candidate_sets.items():
        numeric_columns = STATIC_NUMERIC_COLUMNS + list(dict.fromkeys(extra_columns))
        candidate_numeric_columns[candidate] = numeric_columns
        medians = internal_fit_pd[numeric_columns].median()
        scaler = StandardScaler(with_mean=False)
        fit_numeric = sparse.csr_matrix(scaler.fit_transform(
            internal_fit_pd[numeric_columns].fillna(medians).to_numpy(dtype=float)
        ))
        tuning_numeric = sparse.csr_matrix(scaler.transform(
            internal_tuning_pd[numeric_columns].fillna(medians).to_numpy(dtype=float)
        ))
        fit_matrix = sparse.hstack([cat_fit_matrix, fit_numeric], format='csr')
        tuning_matrix = sparse.hstack([cat_tuning_matrix, tuning_numeric], format='csr')
        ridge = Ridge(alpha=1.0, solver='lsqr').fit(
            fit_matrix, internal_fit_pd[TARGET].to_numpy(dtype=float)
        )
        prediction = ridge.predict(tuning_matrix)
        score = compact_score(segment_metrics(
            internal_tuning_pd[TARGET], prediction, candidate, 'internal_tuning'
        ))
        ablation_rows.append({
            'candidate': candidate, 'numeric_features': len(numeric_columns), **score,
        })
        del fit_numeric, tuning_numeric, fit_matrix, tuning_matrix, ridge
        gc.collect()
    ablation_pd = pd.DataFrame(ablation_rows)
    ablation_global_limit = ablation_pd['global_MAE'].min() + GLOBAL_GUARDRAIL_MINUTES
    ablation_pd['passes_global_guardrail'] = ablation_pd['global_MAE'] <= ablation_global_limit
    ablation_pd = ablation_pd.sort_values(
        ['passes_global_guardrail', 'combined_MAE_score', 'global_MAE'],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    selected_feature_candidate = ablation_pd.iloc[0]['candidate']
    SELECTED_NUMERIC_COLUMNS = candidate_numeric_columns[selected_feature_candidate]
    ablation_pd.to_csv(ABLATION_REPORT, index=False)
    display(ablation_pd)
    print({
        'selected_feature_candidate': selected_feature_candidate,
        'selected_numeric_features': len(SELECTED_NUMERIC_COLUMNS),
    })


,candidate,numeric_features,global_MAE,delayed_MAE,combined_MAE_score,passes_global_guardrail
0,all_without_6h,51,10.266830,18.012386,14.139608,True
1,all_plus_rotation,71,10.246132,18.059455,14.152793,True
2,all_without_1h,51,10.281837,18.063481,14.172659,True
3,airport_route_operator,67,10.255998,18.094775,14.175387,True
4,airport_route,43,10.281733,18.353873,14.317803,True
5,airport,31,10.360174,18.576288,14.468231,True
6,all_without_24h,51,10.356520,18.675862,14.516191,True
7,static_only,7,10.541540,20.682553,15.612047,False


{'selected_feature_candidate': 'all_without_6h', 'selected_numeric_features': 51}


## Ajuste interno y ensemble

Se ajusta `alpha` de Ridge, CatBoost con early stopping y el baseline histórico. Los pesos no
negativos del ensemble se buscan en pasos de 0,1. Ninguna decisión utiliza validation.


In [8]:
if RUN_EXPERIMENT:
    numeric_medians = internal_fit_pd[SELECTED_NUMERIC_COLUMNS].median()
    numeric_scaler = StandardScaler(with_mean=False)
    fit_numeric = sparse.csr_matrix(numeric_scaler.fit_transform(
        internal_fit_pd[SELECTED_NUMERIC_COLUMNS].fillna(numeric_medians).to_numpy(dtype=float)
    ))
    tuning_numeric = sparse.csr_matrix(numeric_scaler.transform(
        internal_tuning_pd[SELECTED_NUMERIC_COLUMNS].fillna(numeric_medians).to_numpy(dtype=float)
    ))
    fit_matrix = sparse.hstack([cat_fit_matrix, fit_numeric], format='csr')
    tuning_matrix = sparse.hstack([cat_tuning_matrix, tuning_numeric], format='csr')
    ridge_rows = []
    ridge_predictions = {}
    ridge_models = {}
    for alpha in (0.1, 1.0, 10.0, 100.0):
        model = Ridge(alpha=alpha, solver='lsqr').fit(
            fit_matrix, internal_fit_pd[TARGET].to_numpy(dtype=float)
        )
        prediction = model.predict(tuning_matrix)
        metrics = segment_metrics(
            internal_tuning_pd[TARGET], prediction, f'ridge_alpha_{alpha:g}', 'internal_tuning'
        )
        ridge_rows.append({'alpha': alpha, **compact_score(metrics)})
        ridge_predictions[alpha] = prediction
        ridge_models[alpha] = model
    ridge_tuning_pd = pd.DataFrame(ridge_rows).sort_values(
        ['combined_MAE_score', 'global_MAE']
    ).reset_index(drop=True)
    selected_alpha = float(ridge_tuning_pd.iloc[0]['alpha'])
    tuning_ridge_prediction = ridge_predictions[selected_alpha]

    from catboost import CatBoostRegressor, Pool
    cat_fit_pd = internal_fit_pd.copy()
    cat_tuning_pd = internal_tuning_pd.copy()
    cat_medians = cat_fit_pd[SELECTED_NUMERIC_COLUMNS].median()
    for frame in (cat_fit_pd, cat_tuning_pd):
        frame[CATEGORICAL_COLUMNS] = frame[CATEGORICAL_COLUMNS].fillna('__MISSING__').astype(str)
        frame[SELECTED_NUMERIC_COLUMNS] = frame[SELECTED_NUMERIC_COLUMNS].fillna(cat_medians)
    CATBOOST_FEATURES = CATEGORICAL_COLUMNS + SELECTED_NUMERIC_COLUMNS
    fit_pool = Pool(
        cat_fit_pd[CATBOOST_FEATURES], label=cat_fit_pd[TARGET],
        cat_features=CATEGORICAL_COLUMNS,
    )
    tuning_pool = Pool(
        cat_tuning_pd[CATBOOST_FEATURES], label=cat_tuning_pd[TARGET],
        cat_features=CATEGORICAL_COLUMNS,
    )
    internal_catboost = CatBoostRegressor(
        loss_function='MAE', eval_metric='MAE', iterations=800, learning_rate=0.03,
        depth=7, l2_leaf_reg=8, boosting_type='Plain', has_time=True,
        one_hot_max_size=10, max_ctr_complexity=2, random_seed=42,
        thread_count=4, od_type='Iter', od_wait=60, allow_writing_files=False,
        verbose=False,
    )
    internal_catboost.fit(fit_pool, eval_set=tuning_pool, use_best_model=True)
    tuning_catboost_prediction = internal_catboost.predict(tuning_pool)
    selected_catboost_iterations = internal_catboost.get_best_iteration() + 1

    internal_baseline = HistoricalMedianBaseline().fit(internal_fit_pd)
    tuning_baseline_prediction = internal_baseline.predict(internal_tuning_pd)
    tuning_components = {
        'ridge': tuning_ridge_prediction,
        'catboost': tuning_catboost_prediction,
        'baseline': tuning_baseline_prediction,
    }
    component_metrics = pd.concat([
        segment_metrics(internal_tuning_pd[TARGET], prediction, name, 'internal_tuning')
        for name, prediction in tuning_components.items()
    ], ignore_index=True)
    ensemble_grid_pd, selected_ensemble = ensemble_grid(
        internal_tuning_pd[TARGET], tuning_components, step=0.1,
        global_guardrail_minutes=GLOBAL_GUARDRAIL_MINUTES,
    )
    selected_weights = {
        name: float(selected_ensemble[f'weight_{name}']) for name in tuning_components
    }
    internal_summary = pd.concat([
        component_metrics,
        segment_metrics(
            internal_tuning_pd[TARGET],
            sum(selected_weights[name] * prediction for name, prediction in tuning_components.items()),
            'selected_ensemble', 'internal_tuning',
        ),
    ], ignore_index=True)
    internal_summary.to_csv(TUNING_REPORT, index=False)
    ensemble_grid_pd.to_csv(ENSEMBLE_REPORT, index=False)
    display(ridge_tuning_pd)
    display(internal_summary[internal_summary['segment'].isin(['all', 'delayed_>15'])])
    print({
        'selected_alpha': selected_alpha,
        'selected_catboost_iterations': selected_catboost_iterations,
        'selected_ensemble_weights': selected_weights,
        'validation_used_for_tuning': False,
    })


,alpha,global_MAE,delayed_MAE,combined_MAE_score
0,10.0,10.060747,18.011332,14.036040
1,1.0,10.266830,18.012386,14.139608
2,100.0,9.929080,18.355580,14.142330
3,0.1,10.324991,18.086450,14.205720


,model,training_scope,segment,rows,MAE,RMSE,median_absolute_error,p90_absolute_error
0,ridge,internal_tuning,all,73275,10.060747,14.949385,7.479426,20.536094
4,ridge,internal_tuning,delayed_>15,15009,18.011332,25.585598,13.969057,34.409375
5,catboost,internal_tuning,all,73275,9.583054,14.735781,6.927037,19.520861
9,catboost,internal_tuning,delayed_>15,15009,19.346039,26.857243,14.993887,36.179096
10,baseline,internal_tuning,all,73275,10.749758,16.389374,7.616667,22.483333
14,baseline,internal_tuning,delayed_>15,15009,24.059643,30.824849,19.600000,42.020000
15,selected_ensemble,internal_tuning,all,73275,9.810266,14.759622,7.224434,19.998116
19,selected_ensemble,internal_tuning,delayed_>15,15009,18.309325,25.869714,14.162978,34.796264


{'selected_alpha': 10.0, 'selected_catboost_iterations': 800, 'selected_ensemble_weights': {'ridge': 0.7, 'catboost': 0.3, 'baseline': 0.0}, 'validation_used_for_tuning': False}


## Reajuste con el 10% completo y evaluación única en validation

Se congelan variables, alpha, iteraciones y pesos. Después se reajustan los tres componentes
con todo el 10% de train y se evalúa validation. El criterio de escalado exige mejorar al menos
0,20 minutos el MAE combinado sin empeorar más de 0,25 el MAE global.


In [9]:
if RUN_EXPERIMENT:
    final_ridge_preprocessor = HashedRidgePreprocessor(
        CATEGORICAL_COLUMNS, SELECTED_NUMERIC_COLUMNS
    )
    full_train_matrix = final_ridge_preprocessor.fit_transform(train_pd)
    validation_matrix = final_ridge_preprocessor.transform(validation_pd)
    final_ridge = Ridge(alpha=selected_alpha, solver='lsqr').fit(
        full_train_matrix, train_pd[TARGET].to_numpy(dtype=float)
    )
    validation_ridge_prediction = final_ridge.predict(validation_matrix)

    final_cat_train_pd = train_pd.copy()
    final_cat_validation_pd = validation_pd.copy()
    final_cat_medians = final_cat_train_pd[SELECTED_NUMERIC_COLUMNS].median()
    for frame in (final_cat_train_pd, final_cat_validation_pd):
        frame[CATEGORICAL_COLUMNS] = frame[CATEGORICAL_COLUMNS].fillna('__MISSING__').astype(str)
        frame[SELECTED_NUMERIC_COLUMNS] = frame[SELECTED_NUMERIC_COLUMNS].fillna(final_cat_medians)
    final_train_pool = Pool(
        final_cat_train_pd[CATBOOST_FEATURES], label=final_cat_train_pd[TARGET],
        cat_features=CATEGORICAL_COLUMNS,
    )
    final_validation_pool = Pool(
        final_cat_validation_pd[CATBOOST_FEATURES], label=final_cat_validation_pd[TARGET],
        cat_features=CATEGORICAL_COLUMNS,
    )
    final_catboost = CatBoostRegressor(
        loss_function='MAE', iterations=selected_catboost_iterations, learning_rate=0.03,
        depth=7, l2_leaf_reg=8, boosting_type='Plain', has_time=True,
        one_hot_max_size=10, max_ctr_complexity=2, random_seed=42,
        thread_count=4, allow_writing_files=False, verbose=False,
    )
    final_catboost.fit(final_train_pool)
    validation_catboost_prediction = final_catboost.predict(final_validation_pool)
    final_baseline = HistoricalMedianBaseline().fit(train_pd)
    validation_baseline_prediction = final_baseline.predict(validation_pd)
    validation_components = {
        'ridge_t60_ops': validation_ridge_prediction,
        'catboost_t60_ops': validation_catboost_prediction,
        'baseline_10pct': validation_baseline_prediction,
    }
    validation_ensemble_prediction = (
        selected_weights['ridge'] * validation_ridge_prediction
        + selected_weights['catboost'] * validation_catboost_prediction
        + selected_weights['baseline'] * validation_baseline_prediction
    )
    validation_components['ensemble_t60_ops'] = validation_ensemble_prediction
    validation_metrics = pd.concat([
        segment_metrics(validation_pd[TARGET], prediction, name, 'deterministic_10pct_train')
        for name, prediction in validation_components.items()
    ], ignore_index=True)
    new_summary_rows = []
    for model_name, model_metrics in validation_metrics.groupby('model'):
        new_summary_rows.append({
            'candidate': model_name, 'training_scope': 'deterministic_10pct_train',
            **compact_score(model_metrics),
        })
    new_summary_pd = pd.DataFrame(new_summary_rows)
    prior_pd = pd.read_csv(REPORT_ROOT / '07_aligned_validation_comparison.csv')
    prior_columns = [
        'candidate', 'training_scope', 'global_MAE', 'delayed_MAE', 'combined_MAE_score'
    ]
    comparison_pd = pd.concat([prior_pd[prior_columns], new_summary_pd], ignore_index=True)
    comparison_global_limit = prior_pd.loc[
        prior_pd['passes_global_MAE_guardrail'].astype(str).str.lower() == 'true', 'global_MAE'
    ].min() + 1.0
    comparison_pd['passes_global_MAE_guardrail'] = comparison_pd['global_MAE'] <= comparison_global_limit
    comparison_pd = comparison_pd.sort_values(
        ['passes_global_MAE_guardrail', 'combined_MAE_score', 'global_MAE'],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    comparison_pd.to_csv(COMPARISON_REPORT, index=False)

    prior_best = prior_pd[
        prior_pd['passes_global_MAE_guardrail'].astype(str).str.lower() == 'true'
    ].sort_values(['combined_MAE_score', 'global_MAE']).iloc[0]
    new_best = new_summary_pd.sort_values(['combined_MAE_score', 'global_MAE']).iloc[0]
    combined_improvement = float(prior_best['combined_MAE_score'] - new_best['combined_MAE_score'])
    global_degradation = float(new_best['global_MAE'] - prior_best['global_MAE'])
    scale_approved = bool(
        combined_improvement >= REQUIRED_COMBINED_IMPROVEMENT
        and global_degradation <= GLOBAL_GUARDRAIL_MINUTES
    )
    scaling_decision = {
        'best_prior_candidate': prior_best['candidate'],
        'best_new_candidate': new_best['candidate'],
        'combined_MAE_improvement_minutes': combined_improvement,
        'global_MAE_degradation_minutes': global_degradation,
        'required_combined_improvement_minutes': REQUIRED_COMBINED_IMPROVEMENT,
        'maximum_global_degradation_minutes': GLOBAL_GUARDRAIL_MINUTES,
        'scale_beyond_10pct': scale_approved,
        'test_processed_read': False,
    }
    DECISION_PATH.write_text(json.dumps(scaling_decision, indent=2), encoding='utf-8')
    pd.DataFrame({
        'ECTRL ID': validation_pd['ECTRL ID'].to_numpy(),
        TARGET: validation_pd[TARGET].to_numpy(),
        'ridge_t60_ops': validation_ridge_prediction,
        'catboost_t60_ops': validation_catboost_prediction,
        'baseline_10pct': validation_baseline_prediction,
        'ensemble_t60_ops': validation_ensemble_prediction,
    }).to_parquet(PREDICTIONS_PATH, index=False)
    final_catboost.save_model(str(MODELS_ROOT / '08_catboost_t60_ops_10pct.cbm'))
    joblib.dump(
        {'model': final_ridge, 'preprocessor': final_ridge_preprocessor,
         'numeric_columns': SELECTED_NUMERIC_COLUMNS,
         'categorical_columns': CATEGORICAL_COLUMNS},
        MODELS_ROOT / '08_ridge_t60_ops_10pct.joblib',
    )
    joblib.dump(final_baseline, MODELS_ROOT / '08_baseline_10pct.joblib')
    display(comparison_pd.head(12))
    print(scaling_decision)
    print({
        'selected_features': selected_feature_candidate,
        'ensemble_weights': selected_weights,
        'test_processed_read': False,
    })


,candidate,training_scope,global_MAE,delayed_MAE,combined_MAE_score,passes_global_MAE_guardrail
0,ridge_t60_ops,deterministic_10pct_train,9.940508,18.207436,14.073972,True
1,ensemble_t60_ops,deterministic_10pct_train,9.716095,18.527837,14.121966,True
2,catboost_t60_ops,deterministic_10pct_train,9.537456,19.634829,14.586143,True
3,linear_regression / original,deterministic_1pct_train,10.871172,19.267681,15.069427,True
4,linear_regression / log,deterministic_1pct_train,10.875577,19.322786,15.099181,True
5,linear_regression / yeo_johnson,deterministic_1pct_train,10.874277,19.338263,15.106270,True
6,catboost_g1.00_p10 / native_categorical_t60,deterministic_10pct_train,9.983605,20.483587,15.233596,True
7,catboost_g1.00_p05 / native_categorical_t60,deterministic_5pct_train,10.058765,20.601351,15.330058,True
8,historical_route_airline_fallback / not_applic...,full_train,9.938292,21.108836,15.523564,True
9,catboost_g1.00_p01 / native_categorical_t60,deterministic_1pct_train,10.369805,21.029858,15.699831,True


{'best_prior_candidate': 'linear_regression / original', 'best_new_candidate': 'ridge_t60_ops', 'combined_MAE_improvement_minutes': 0.995454376458623, 'global_MAE_degradation_minutes': -0.9306637041817662, 'required_combined_improvement_minutes': 0.2, 'maximum_global_degradation_minutes': 0.25, 'scale_beyond_10pct': True, 'test_processed_read': False}
{'selected_features': 'all_without_6h', 'ensemble_weights': {'ridge': 0.7, 'catboost': 0.3, 'baseline': 0.0}, 'test_processed_read': False}


## Conclusiones observadas

- La ablacion selecciona todas las entidades y rotacion, pero elimina la ventana de 6h; conserva 1h y 24h.
- Ridge T-60 obtiene MAE global 9.941, MAE retrasados 18.207 y combinado 14.074.
- El ensemble elegido en tuning usa 70% Ridge y 30% CatBoost, pero Ridge generaliza mejor en validation.
- Frente al mejor candidato previo, Ridge mejora el combinado en 0.995 minutos y el global en 0.931.
- Se aprueba escalar mas alla del 10% bajo el criterio congelado; test sigue sin leer.


## Guardrails de interpretación

- Los vuelos retrasados son un segmento de evaluación; su estado no se conoce a T−60.
- `AC Registration` se asume asignada a T−60; la cobertura real se informa.
- Un vuelo histórico solo aporta salida tras su off-block real y llegada tras su llegada real.
- Las variables, hiperparámetros y pesos se seleccionan en tuning interno, no en validation.
- Validation se usa una vez; el Parquet test y marzo de 2023 no se leen.
- El escalado más allá del 10% depende del criterio cuantitativo congelado.
